# Quartet multiomics preprocessing and EDA

Load one Figshare matrix, align it to the matching metadata, inspect the batch/condition structure, and plot the uncorrected data. Change `DATASET_ID` to switch between full, balanced, and confounded datasets.

In [ ]:
library(tidyverse)
library(readxl)

WORKDIR <- if (dir.exists("figshare_data")) "." else "evaluation_data/multiomics"
DATA_DIR <- file.path(WORKDIR, "figshare_data")
PLOTS_DIR <- file.path(WORKDIR, "plots")
dir.create(PLOTS_DIR, showWarnings = FALSE, recursive = TRUE)

source(file.path(WORKDIR, "..", "..", "evaluation_utils", "filtering.R"))

# Main options.
# DATASET_ID controls the single-dataset preview cells.
# DATASET_IDS controls the batch plot generation cell near the end.
# Examples: Proteomics_full, Transcriptomics_full, Metabolomics_full,
# Proteomics_balanced, Proteomics_confounded, etc.
DATASET_ID <- "Transcriptomics_full"
DATASET_IDS <- c("Transcriptomics_full", "Proteomics_full", "Metabolomics_full")
TOP_VARIABLE_FEATURES <- 3000
BOXPLOT_MAX_FEATURES <- 5000
WRITE_ALIGNED_INPUTS <- FALSE


## Dataset registry

In [ ]:
dataset_registry <- tribble(
  ~dataset_id, ~omics, ~scenario, ~file_name, ~scale, ~metadata_source,
  "Transcriptomics_full", "Transcriptomics", "Full", "Transcriptomics_fulldataset_log2FPKM_r26907c180.csv", "log2(FPKM + 0.01)", "full",
  "Transcriptomics_balanced", "Transcriptomics", "Balanced", "Transcriptomics_Balanced_log2FPKM_r26907c45.csv", "log2(FPKM + 0.01)", "scenario",
  "Transcriptomics_confounded", "Transcriptomics", "Confounded", "Transcriptomics_Confounded_log2FPKM_r26907c45.csv", "log2(FPKM + 0.01)", "scenario",
  "Proteomics_full", "Proteomics", "Full", "Proteomics_fulldataset_log2FOT_r3489c180.csv", "log2(FOT + 0.01)", "full",
  "Proteomics_balanced", "Proteomics", "Balanced", "Proteomics_Balanced_log2FOT_r3489c45.csv", "log2(FOT + 0.01)", "scenario",
  "Proteomics_confounded", "Proteomics", "Confounded", "Proteomics_Confounded_log2FOT_r3489c45.csv", "log2(FOT + 0.01)", "scenario",
  "Metabolomics_full", "Metabolomics", "Full", "Metabolomics_fulldataset_log2expr_r71c180.csv", "log2(expr + 1)", "full",
  "Metabolomics_balanced", "Metabolomics", "Balanced", "Metabolomics_Balanced_log2expr_r71c45.csv", "log2(expr + 1)", "scenario",
  "Metabolomics_confounded", "Metabolomics", "Confounded", "Metabolomics_Confounded_log2expr_r71c45.csv", "log2(expr + 1)", "scenario"
)

dataset_registry

selected <- dataset_registry %>% filter(dataset_id == DATASET_ID)
if (nrow(selected) != 1) {
  stop("Unknown DATASET_ID: ", DATASET_ID, call. = FALSE)
}
selected


## Loading helpers

The proteomics balanced/confounded CSV files use periods in a few instrument tokens, while the metadata uses hyphens. `normalize_library_ids()` applies the same convention to both sides before joining.

In [ ]:
normalize_datatype <- function(x) {
  dplyr::recode(
    as.character(x),
    "RNA" = "Transcriptomics",
    "Protein" = "Proteomics",
    "Metabolite" = "Metabolomics",
    .default = as.character(x)
  )
}

normalize_library_ids <- function(x, omics) {
  x <- as.character(x)
  if (omics == "Proteomics") {
    stringr::str_replace_all(x, stringr::fixed("."), "-")
  } else {
    x
  }
}

load_expression_matrix <- function(path, omics) {
  raw <- readr::read_csv(path, show_col_types = FALSE)
  feature_id_col <- names(raw)[1]
  feature_cols <- feature_id_col
  feature_ids <- as.character(raw[[feature_id_col]])

  if ("metabolite_name" %in% names(raw)) {
    feature_cols <- c(feature_cols, "metabolite_name")
    feature_ids <- paste(feature_ids, raw[["metabolite_name"]], sep = "|")
  }

  expr <- raw %>% select(-all_of(feature_cols)) %>% as.data.frame(check.names = FALSE)
  rownames(expr) <- make.unique(feature_ids)
  colnames(expr) <- normalize_library_ids(colnames(expr), omics)
  expr[] <- lapply(expr, as.numeric)
  as.matrix(expr)
}

load_full_metadata <- function(path) {
  readr::read_csv(path, show_col_types = FALSE) %>%
    mutate(datatype = normalize_datatype(datatype)) %>%
    group_by(datatype) %>%
    mutate(batch_code = sprintf("B%02d", dense_rank(batch))) %>%
    ungroup() %>%
    transmute(
      file = library,
      condition = sample,
      batch = batch,
      batch_code = batch_code,
      lab = lab,
      platform = platform,
      protocol = replace_na(as.character(protocol), "not_available"),
      datatype = datatype,
      rep = as.integer(rep),
      date = as.character(date),
      role = "all",
      scenario = "Full"
    )
}

load_scenario_metadata <- function(path) {
  readxl::read_excel(path, sheet = "metadata") %>%
    transmute(
      file = Library,
      condition = `Sample group`,
      batch = Batch,
      batch_code = `Batch code`,
      lab = Lab,
      platform = Platform,
      protocol = "not_available",
      datatype = `Data type`,
      rep = as.integer(Replicate),
      date = as.character(Date),
      role = `Study or reference sample`,
      scenario = Scenario
    )
}

load_metadata_for_selection <- function(selected_row) {
  if (selected_row$metadata_source == "full") {
    metadata <- load_full_metadata(file.path(DATA_DIR, "meta_full_dataset_3omics.csv")) %>%
      filter(datatype == selected_row$omics)
  } else {
    metadata <- load_scenario_metadata(file.path(DATA_DIR, "meta_balancedconfounded_dataset_3omics.xlsx")) %>%
      filter(datatype == selected_row$omics, scenario == selected_row$scenario, role == "study")
  }

  metadata %>%
    mutate(file = normalize_library_ids(file, selected_row$omics)) %>%
    arrange(batch_code, condition, rep)
}

align_expression_to_metadata <- function(expr, metadata) {
  missing_in_matrix <- setdiff(metadata$file, colnames(expr))
  extra_in_matrix <- setdiff(colnames(expr), metadata$file)

  if (length(missing_in_matrix) > 0) {
    stop(
      "Metadata samples missing from expression matrix: ",
      paste(head(missing_in_matrix, 10), collapse = ", "),
      call. = FALSE
    )
  }
  if (length(extra_in_matrix) > 0) {
    warning("Expression matrix has samples absent from metadata: ", paste(head(extra_in_matrix, 10), collapse = ", "))
  }

  metadata <- metadata %>% filter(file %in% colnames(expr)) %>% arrange(match(file, colnames(expr)))
  expr <- expr[, metadata$file, drop = FALSE]
  list(expr = expr, metadata = metadata)
}


## Load and align selected dataset

In [ ]:
expr <- load_expression_matrix(file.path(DATA_DIR, selected$file_name), selected$omics)
metadata <- load_metadata_for_selection(selected)
aligned <- align_expression_to_metadata(expr, metadata)
expr <- aligned$expr
metadata <- aligned$metadata

cat("Dataset:", DATASET_ID, "\n")
cat("Omics:", selected$omics, "\n")
cat("Scenario:", selected$scenario, "\n")
cat("Scale:", selected$scale, "\n")
cat("Matrix:", nrow(expr), "features x", ncol(expr), "samples\n")
cat("Metadata:", nrow(metadata), "rows x", ncol(metadata), "columns\n")

metadata %>% glimpse()


## Metadata and batch structure

In [ ]:
metadata %>% count(condition, name = "n_samples")
metadata %>% count(batch_code, batch, condition) %>% pivot_wider(names_from = condition, values_from = n, values_fill = 0)
metadata %>% count(lab, platform, batch, name = "n_samples") %>% arrange(lab, platform, batch)

fedrbe_effective_min_samples <- fedrbe_min_samples(
  num_batches = n_distinct(metadata$batch),
  covariates = setdiff(sort(unique(metadata$condition)), "D5"),
  min_samples = 0
)
cat("Batches:", n_distinct(metadata$batch), "\n")
cat("Labs:", n_distinct(metadata$lab), "\n")
cat("Platforms:", n_distinct(metadata$platform), "\n")
cat("FedRBE-style minimum samples for", n_distinct(metadata$batch), "batches and condition covariates:", fedrbe_effective_min_samples, "\n")


## Expression summaries

In [ ]:
sample_stats <- tibble(
  file = colnames(expr),
  missing_values = colSums(is.na(expr)),
  missing_fraction = colMeans(is.na(expr)),
  median = apply(expr, 2, median, na.rm = TRUE),
  mean = colMeans(expr, na.rm = TRUE),
  iqr = apply(expr, 2, IQR, na.rm = TRUE)
) %>% left_join(metadata, by = "file")

feature_stats <- tibble(
  feature = rownames(expr),
  missing_fraction = rowMeans(is.na(expr)),
  variance = apply(expr, 1, var, na.rm = TRUE)
)

sample_stats %>% summary()
feature_stats %>% summarize(
  n_features = n(),
  n_complete = sum(missing_fraction == 0),
  median_missing_fraction = median(missing_fraction),
  max_missing_fraction = max(missing_fraction),
  n_zero_or_na_variance = sum(!is.finite(variance) | variance == 0)
)


## PCA before correction

PCA uses feature-wise centering/scaling. To keep transcriptomics responsive, it uses the top variable features by default.

In [ ]:
prepare_pca <- function(expr, metadata, top_n = 3000) {
  feature_var <- apply(expr, 1, var, na.rm = TRUE)
  keep <- which(is.finite(feature_var) & feature_var > 0)
  keep <- keep[order(feature_var[keep], decreasing = TRUE)]
  keep <- head(keep, min(top_n, length(keep)))
  if (length(keep) < 2) {
    stop("Need at least two variable features for PCA.", call. = FALSE)
  }

  x <- t(expr[keep, , drop = FALSE])
  if (anyNA(x)) {
    for (j in seq_len(ncol(x))) {
      med <- median(x[, j], na.rm = TRUE)
      if (!is.finite(med)) med <- 0
      x[is.na(x[, j]), j] <- med
    }
  }

  pca <- prcomp(x, center = TRUE, scale. = TRUE)
  variance <- pca$sdev^2 / sum(pca$sdev^2)
  scores <- as_tibble(pca$x[, 1:2], rownames = "file") %>% left_join(metadata, by = "file")
  list(scores = scores, variance = variance, n_features = length(keep))
}

pca_res <- prepare_pca(expr, metadata, TOP_VARIABLE_FEATURES)
cat("PCA features used:", pca_res$n_features, "\n")

pca_batch <- ggplot(pca_res$scores, aes(PC1, PC2, color = batch, shape = condition)) +
  geom_point(size = 3, alpha = 0.9) +
  theme_classic() +
  theme(legend.text = element_text(size = 8), legend.title = element_text(size = 10)) +
  labs(
    title = paste(DATASET_ID, "before correction: batch"),
    x = paste0("PC1 [", round(pca_res$variance[1] * 100, 1), "%]"),
    y = paste0("PC2 [", round(pca_res$variance[2] * 100, 1), "%]")
  )

pca_condition <- ggplot(pca_res$scores, aes(PC1, PC2, color = condition)) +
  geom_point(size = 3, alpha = 0.9) +
  theme_classic() +
  labs(
    title = paste(DATASET_ID, "before correction: condition"),
    x = paste0("PC1 [", round(pca_res$variance[1] * 100, 1), "%]"),
    y = paste0("PC2 [", round(pca_res$variance[2] * 100, 1), "%]")
  )

pca_batch
pca_condition

ggsave(file.path(PLOTS_DIR, paste0(DATASET_ID, "_pca_by_batch.png")), pca_batch, width = 9.5, height = 6.5, bg = "white")
ggsave(file.path(PLOTS_DIR, paste0(DATASET_ID, "_pca_by_condition.png")), pca_condition, width = 6, height = 5, bg = "white")


## Boxplots and density before correction

In [ ]:
set.seed(1)
box_features <- if (nrow(expr) > BOXPLOT_MAX_FEATURES) sample(rownames(expr), BOXPLOT_MAX_FEATURES) else rownames(expr)

long_expr <- expr[box_features, , drop = FALSE] %>%
  as.data.frame(check.names = FALSE) %>%
  rownames_to_column("feature") %>%
  pivot_longer(-feature, names_to = "file", values_to = "value") %>%
  left_join(metadata, by = "file")

box_by_sample <- ggplot(long_expr, aes(x = file, y = value, fill = batch)) +
  geom_boxplot(outlier.shape = NA) +
  theme_minimal() +
  theme(axis.text.x = element_blank(), axis.ticks.x = element_blank()) +
  labs(title = paste(DATASET_ID, "sample distributions before correction"), x = "sample", y = selected$scale)

box_by_batch <- ggplot(long_expr, aes(x = batch, y = value, fill = batch)) +
  geom_boxplot(outlier.shape = NA) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1), legend.position = "none") +
  labs(title = paste(DATASET_ID, "batch distributions before correction"), x = "batch", y = selected$scale)

density_by_batch <- ggplot(long_expr, aes(x = value, color = batch)) +
  geom_density(na.rm = TRUE) +
  theme_minimal() +
  theme(legend.text = element_text(size = 8), legend.title = element_text(size = 10)) +
  labs(title = paste(DATASET_ID, "intensity density before correction"), x = selected$scale, y = "density")

box_by_sample
box_by_batch
density_by_batch

ggsave(file.path(PLOTS_DIR, paste0(DATASET_ID, "_boxplot_by_sample.png")), box_by_sample, width = 10, height = 5, bg = "white")
ggsave(file.path(PLOTS_DIR, paste0(DATASET_ID, "_boxplot_by_batch.png")), box_by_batch, width = 10, height = 5, bg = "white")
ggsave(file.path(PLOTS_DIR, paste0(DATASET_ID, "_density_by_batch.png")), density_by_batch, width = 9.5, height = 6, bg = "white")


## Generate plots for selected multiomics datasets

The matrices are plotted per omics layer because transcriptomics, proteomics, and metabolomics have different feature spaces and measurement scales. `DATASET_IDS` in the first cell controls which layers/scenarios are generated.

In [ ]:
save_dataset_plots <- function(dataset_id) {
  selected_row <- dataset_registry %>% filter(.data$dataset_id == .env$dataset_id)
  if (nrow(selected_row) != 1) {
    stop("Unknown DATASET_ID: ", dataset_id, call. = FALSE)
  }

  expr_i <- load_expression_matrix(file.path(DATA_DIR, selected_row$file_name), selected_row$omics)
  metadata_i <- load_metadata_for_selection(selected_row)
  aligned_i <- align_expression_to_metadata(expr_i, metadata_i)
  expr_i <- aligned_i$expr
  metadata_i <- aligned_i$metadata

  pca_i <- prepare_pca(expr_i, metadata_i, TOP_VARIABLE_FEATURES)
  pca_batch_i <- ggplot(pca_i$scores, aes(PC1, PC2, color = batch, shape = condition)) +
    geom_point(size = 3, alpha = 0.9) +
    theme_classic() +
    theme(legend.text = element_text(size = 8), legend.title = element_text(size = 10)) +
    labs(
      title = paste(dataset_id, "before correction: batch"),
      x = paste0("PC1 [", round(pca_i$variance[1] * 100, 1), "%]"),
      y = paste0("PC2 [", round(pca_i$variance[2] * 100, 1), "%]")
    )

  pca_condition_i <- ggplot(pca_i$scores, aes(PC1, PC2, color = condition)) +
    geom_point(size = 3, alpha = 0.9) +
    theme_classic() +
    labs(
      title = paste(dataset_id, "before correction: condition"),
      x = paste0("PC1 [", round(pca_i$variance[1] * 100, 1), "%]"),
      y = paste0("PC2 [", round(pca_i$variance[2] * 100, 1), "%]")
    )

  ggsave(file.path(PLOTS_DIR, paste0(dataset_id, "_pca_by_batch.png")), pca_batch_i, width = 9.5, height = 6.5, bg = "white")
  ggsave(file.path(PLOTS_DIR, paste0(dataset_id, "_pca_by_condition.png")), pca_condition_i, width = 6, height = 5, bg = "white")

  set.seed(1)
  box_features_i <- if (nrow(expr_i) > BOXPLOT_MAX_FEATURES) sample(rownames(expr_i), BOXPLOT_MAX_FEATURES) else rownames(expr_i)
  long_expr_i <- expr_i[box_features_i, , drop = FALSE] %>%
    as.data.frame(check.names = FALSE) %>%
    rownames_to_column("feature") %>%
    pivot_longer(-feature, names_to = "file", values_to = "value") %>%
    left_join(metadata_i, by = "file")

  box_by_sample_i <- ggplot(long_expr_i, aes(x = file, y = value, fill = batch)) +
    geom_boxplot(outlier.shape = NA) +
    theme_minimal() +
    theme(axis.text.x = element_blank(), axis.ticks.x = element_blank()) +
    labs(title = paste(dataset_id, "sample distributions before correction"), x = "sample", y = selected_row$scale)

  box_by_batch_i <- ggplot(long_expr_i, aes(x = batch, y = value, fill = batch)) +
    geom_boxplot(outlier.shape = NA) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1), legend.position = "none") +
    labs(title = paste(dataset_id, "batch distributions before correction"), x = "batch", y = selected_row$scale)

  density_by_batch_i <- ggplot(long_expr_i, aes(x = value, color = batch)) +
    geom_density(na.rm = TRUE) +
    theme_minimal() +
    theme(legend.text = element_text(size = 8), legend.title = element_text(size = 10)) +
    labs(title = paste(dataset_id, "intensity density before correction"), x = selected_row$scale, y = "density")

  ggsave(file.path(PLOTS_DIR, paste0(dataset_id, "_boxplot_by_sample.png")), box_by_sample_i, width = 10, height = 5, bg = "white")
  ggsave(file.path(PLOTS_DIR, paste0(dataset_id, "_boxplot_by_batch.png")), box_by_batch_i, width = 10, height = 5, bg = "white")
  ggsave(file.path(PLOTS_DIR, paste0(dataset_id, "_density_by_batch.png")), density_by_batch_i, width = 9.5, height = 6, bg = "white")

  tibble(
    dataset_id = dataset_id,
    omics = selected_row$omics,
    scenario = selected_row$scenario,
    features = nrow(expr_i),
    samples = ncol(expr_i),
    batches = n_distinct(metadata_i$batch)
  )
}

plot_summaries <- purrr::map_dfr(DATASET_IDS, save_dataset_plots)
plot_summaries


## Optional aligned exports

Set `WRITE_ALIGNED_INPUTS <- TRUE` in the first code cell to write the selected, metadata-aligned matrix and design table. This is intentionally off by default so EDA does not create new batch-correction inputs until the target omics/scenario is chosen.

In [ ]:
if (WRITE_ALIGNED_INPUTS) {
  export_dir <- file.path(WORKDIR, "before", DATASET_ID)
  dir.create(export_dir, showWarnings = FALSE, recursive = TRUE)

  readr::write_tsv(metadata, file.path(export_dir, "design.tsv"))
  expr %>%
    as.data.frame(check.names = FALSE) %>%
    rownames_to_column("feature") %>%
    readr::write_tsv(file.path(export_dir, "intensities.tsv"))

  cat("Wrote aligned EDA inputs to", export_dir, "\n")
}


In [ ]:
sessionInfo()
